# SafeLens Phase 4 -- DeBERTa training on Colab GPU

Local M2 (MPS) training was measured at ~370-524 seconds/step (projected 53-76 hours for 525 steps) -- infeasible. This notebook runs the **exact same, unmodified** `scripts/train_deberta.py` used locally, on a Colab GPU. Device selection is automatic (`safelens.utils.device.detect_device` picks CUDA when available), so no code changes are needed for a fair, reproducible comparison against the frozen Phase 3 baseline.

**Before running:** Runtime -> Change runtime type -> GPU (T4 or better).

**You must provide two things this notebook cannot fetch on its own:**
1. A GitHub token with read access to the private `SafeLens` repo, stored as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar).
2. The exact Phase 2 processed split files (`train.jsonl`, `validation.jsonl`, `test.jsonl`), zipped from your local `data/processed/civil_comments/` directory and uploaded here. These are gitignored (not in the repo) specifically so the frozen split is never silently regenerated -- uploading the exact local files guarantees zero drift from what Phase 3 was evaluated against.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/saitejasrivilli/SafeLens.git
%cd SafeLens

## Upload the exact Phase 2 processed split

On your machine: `cd SafeLens/data/processed/civil_comments && zip civil_comments_processed.zip train.jsonl validation.jsonl test.jsonl`

Then upload that zip in the next cell.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select civil_comments_processed.zip
!mkdir -p data/processed/civil_comments
!unzip -o civil_comments_processed.zip -d data/processed/civil_comments
!ls -la data/processed/civil_comments

In [ ]:
!pip install -q -e ".[dev]"

In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
print('device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Run the exact same training script used locally

No code changes. `detect_device()` will select CUDA automatically.

In [ ]:
!python scripts/train_deberta.py

## Package results for download

Two archives: a small one (experiment.json, plots, config/metadata -- safe to commit) and a large one (trained model weights -- for local use only, still not committed to Git per Phase 4 instructions).

In [ ]:
!cd benchmarks/results/deberta && zip -r /content/deberta_benchmark_results.zip .
!cd models/text/deberta && zip -r /content/deberta_model_artifacts.zip v1
from google.colab import files
files.download('/content/deberta_benchmark_results.zip')
files.download('/content/deberta_model_artifacts.zip')

## After downloading

Locally: unzip `deberta_benchmark_results.zip` into `benchmarks/results/deberta/` and `deberta_model_artifacts.zip` into `models/text/deberta/` (the latter stays gitignored). Then continue the Phase 4 write-up (docs, comparison table, error analysis) from the real `experiment.json` produced here -- do not fabricate numbers if this run's results differ from any earlier expectation.